# OpticksHits on any GDML geometry (DUNE, ICARUS, ...)

General version of `OpticksHits_heatmap.ipynb` for detectors that are **not cylinders**: planes of
photon detectors in LAr TPCs (DUNE FD arapucas, ICARUS PMTs, ProtoDUNE, ...).

| Function | What it does |
|---|---|
| `read_gdml_sensors(gdml, select=...)` | Walks the full GDML tree and returns **every placement** of the sensor volumes with a unique `sensor_index`, its tree path, copy number, global centre and global bounding box |
| `read_gdml_volumes(gdml, regex)` | Global bounding boxes of any other volumes (cryostat, TPCs, ...) to draw as context |
| `find_planes(sensors)` | Groups sensors into planes (same thin axis, same coordinate) |
| `draw_sensor_positions(sensors)` | 3D view + XY / XZ / ZY projections with every sensor's footprint |
| `draw_sensor_planes(sensors)` | One panel per plane, sensors drawn at their real size |
| `load_hits(root_file)` | `OpticksHits` (or `Geant4Hits`) → pandas DataFrame |
| `match_hits(hits, sensors)` | Assigns every hit to the sensor whose bounding box it is in / closest to |
| `plot_hit_heatmap_planes(sensors, counts)` | Hits per sensor, one panel per plane |
| `plot_hit_heatmap_3d(sensors, counts, style="face")` | 3D heat map, each sensor drawn as its real rectangle (`"face"`) or full box (`"box"`) |

## Configuration
Use **absolute paths** (`~` and `$VARS` are expanded); relative paths are resolved from the directory
Jupyter was started in.

In [ ]:
GDML_FILE = "dune10kt_v6_refactored_nowires_NoTPCRegions.gdml"
ROOT_FILE = "test_dunehd_10k.root"   # gdml_det output
TREE_NAME = "OpticksHits"   # or "Geant4Hits"

SENSORS   = "aux"           # "aux" = LVs tagged SensDet/PD = PhotonDetector (what OpticalSims uses),
                            # or a regex on LV names, e.g. r"^volOpDetSensitive$" for ICARUS
CONTEXT   = r"^volTPCActive"  # regex of LVs drawn as outlines (None = nothing)
MATCH     = "position"      # "position" (hit x,y,z -> sensor) or "id" (SensorID == sensor copy number)
EVENTS    = None            # None = all, int, or list of evtIDs
LOG_COLOR = True
LENGTH_UNIT = "m"           # plot axes in "m", "cm" or "mm"
VERTICAL_AXIS = "y"         # axis drawn upwards in 3D views: "y" for LArTPCs (DUNE, ICARUS), "z" for SK-like tanks
SAVE_DIR  = "plots/dune"            # absolute path to also save PNGs
# Figures embedded in the notebook (keeps the .ipynb small: JupyterHub proxies often reject
# saves above ~1 MB with "413 Request Entity Too Large"). Files in SAVE_DIR are always full-quality PNG.
INLINE_FORMAT = "png"      # "jpeg" (small), "png" (sharper, ~5x bigger) or None (show nothing inline; use SAVE_DIR)
INLINE_DPI    = 100          # resolution of the embedded figures
EVENT_DISPLAYS = 1         # how many single-event heat maps to draw in section 4

In [ ]:
import os
from pathlib import Path


def _find_by_name(name, roots, max_depth=5, limit=10):
    hits = []
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        base = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).parts) - base >= max_depth:
                dirnames[:] = []
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if name in filenames:
                hits.append(Path(dirpath) / name)
                if len(hits) >= limit:
                    return hits
    return hits


def absolute_path(p, what="file", must_exist=True):
    # Expand ~ / $VARS, require an absolute path, and (optionally) check that it exists.
    p = Path(os.path.expandvars(os.path.expanduser(str(p))))
    if not p.is_absolute():
        print(f"WARNING: {what} path {str(p)!r} is relative; resolving it from {Path.cwd()}")
        p = Path.cwd() / p
    p = p.resolve()
    if must_exist and not p.exists():
        found = list(dict.fromkeys(_find_by_name(p.name, [Path.home(), p.parent.parent])))
        hint = ("\n  files with that name:\n    " + "\n    ".join(map(str, found))) if found else \
               f"\n  no file called {p.name!r} found under {Path.home()}"
        raise FileNotFoundError(f"{what} not found: {p}{hint}")
    return p


GDML_FILE = absolute_path(GDML_FILE, "GDML file")
ROOT_FILE = absolute_path(ROOT_FILE, "ROOT file")
SAVE_DIR = absolute_path(SAVE_DIR, "SAVE_DIR", must_exist=False) if SAVE_DIR else None
print("GDML_FILE =", GDML_FILE)
print("ROOT_FILE =", ROOT_FILE)

## Imports

In [ ]:
import math
import re
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LogNorm, Normalize
from matplotlib.patches import Rectangle
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.spatial import cKDTree
import uproot

print("uproot", uproot.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

plt.rcParams["figure.dpi"] = INLINE_DPI
if INLINE_FORMAT == "jpeg":
    set_matplotlib_formats("jpeg", pil_kwargs={"quality": 70, "optimize": True})
elif INLINE_FORMAT == "png":
    set_matplotlib_formats("png")
else:                                   # nothing embedded: figures only go to SAVE_DIR
    plt.ioff()
    set_matplotlib_formats("png")
print(f"inline figures: {INLINE_FORMAT or 'off'} @ {INLINE_DPI} dpi")

## GDML reader
Placements follow Geant4's GDML reader: physvol rotation angles build `M = Rz·Ry·Rx` and the placement
uses `M⁻¹`; transforms are composed down the tree, so a volume placed N times through its ancestors
appears N times. Each sensor gets a **global axis-aligned bounding box** from its solid
(box, tube, cone, sphere, orb, ellipsoid, eltube, trd, trap, polycone, polyhedra, para, torus,
and union / subtraction / intersection of those).

In [ ]:
_UNITS = {"mm": 1.0, "cm": 10.0, "m": 1000.0, "km": 1e6, "um": 1e-3, "nm": 1e-6,
          "rad": 1.0, "mrad": 1e-3, "deg": math.pi / 180.0, "degree": math.pi / 180.0}


class _Expr:
    # Evaluates GDML numeric attributes ("16900", "2*R", "180*deg", "pi/2") with <define> constants.
    def __init__(self):
        self.ns = {"pi": math.pi, "e": math.e, "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos,
                   "tan": math.tan, "asin": math.asin, "acos": math.acos, "atan": math.atan,
                   "atan2": math.atan2, "exp": math.exp, "log": math.log, "abs": abs, "pow": pow}
        self.ns.update(_UNITS)

    def __call__(self, s, default=0.0):
        if s is None:
            return default
        try:
            return float(s)
        except ValueError:
            return float(eval(str(s).replace("^", "**"), {"__builtins__": {}}, self.ns))

    def define(self, name, value):
        try:
            self.ns[name] = self(value)
        except Exception:
            pass                       # matrices / unresolved expressions are not needed here


def _rot_matrix(ax, ay, az):
    # CLHEP: rot.rotateX(ax); rot.rotateY(ay); rot.rotateZ(az)  ->  M = Rz @ Ry @ Rx
    cx, sx, cy, sy, cz, sz = math.cos(ax), math.sin(ax), math.cos(ay), math.sin(ay), math.cos(az), math.sin(az)
    Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
    Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
    Rz = np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]])
    return Rz @ Ry @ Rx


def _box_corners(lo, hi):
    return np.array([[x, y, z] for x in (lo[0], hi[0]) for y in (lo[1], hi[1]) for z in (lo[2], hi[2])])


class GDML:
    # Parsed GDML file: defines, solids (local bounding boxes), volumes, world name.
    def __init__(self, path):
        self.path = str(path)
        root = ET.parse(self.path).getroot()
        for el in root.iter():                         # strip XML namespaces
            if isinstance(el.tag, str) and "}" in el.tag:
                el.tag = el.tag.split("}", 1)[1]
        self.root, self.ev = root, _Expr()
        self.positions, self.rotations = {}, {}
        for d in root.findall("define"):
            for el in d:
                name = el.get("name")
                if el.tag in ("constant", "variable"):
                    self.ev.define(name, el.get("value"))
                elif el.tag == "quantity":
                    self.ev.define(name, f"({el.get('value')})*{el.get('unit', '1')}")
                elif el.tag == "position":
                    self.positions[name] = el
                elif el.tag == "rotation":
                    self.rotations[name] = el
        self.solids = {}
        for s in root.findall("solids"):
            for el in s:
                if el.get("name"):
                    self.solids[el.get("name")] = el
        st = root.find("structure")
        self.volumes = {v.get("name"): v for v in st if v.tag in ("volume", "assembly")}
        self.world = root.find("setup/world").get("ref")
        self._bbox_cache = {}
        self.unknown_solids = set()

    # -- placements ---------------------------------------------------------
    def vec(self, el, kind):
        if el is None:
            return np.zeros(3)
        u = _UNITS.get(el.get("unit", "mm" if kind == "pos" else "rad"), 1.0)
        return np.array([self.ev(el.get(k), 0.0) for k in "xyz"]) * u

    def placement(self, node):
        # (translation, rotation matrix) of a physvol / boolean 'second' as Geant4 applies it.
        pos = node.find("position")
        if pos is None and node.find("positionref") is not None:
            pos = self.positions.get(node.find("positionref").get("ref"))
        rot = node.find("rotation")
        if rot is None and node.find("rotationref") is not None:
            rot = self.rotations.get(node.find("rotationref").get("ref"))
        return self.vec(pos, "pos"), _rot_matrix(*self.vec(rot, "rot")).T

    # -- solids -------------------------------------------------------------
    def bbox(self, name):
        # Local bounding box (lo, hi) of solid `name` in mm.
        if name in self._bbox_cache:
            return self._bbox_cache[name]
        s = self.solids.get(name)
        if s is None:
            self.unknown_solids.add(name)
            lo, hi = -np.ones(3) * 5, np.ones(3) * 5
        else:
            lo, hi = self._solid_bbox(s)
        self._bbox_cache[name] = (lo, hi)
        return lo, hi

    def _solid_bbox(self, s):
        g = lambda k, d=0.0: self.ev(s.get(k), d)
        L = _UNITS.get(s.get("lunit", "mm"), 1.0)
        t = s.tag
        if t in ("union", "subtraction", "intersection"):
            lo1, hi1 = self.bbox(s.find("first").get("ref"))
            if t != "union":
                return lo1, hi1                       # result is inside the first solid
            lo2, hi2 = self.bbox(s.find("second").get("ref"))
            tr, R = self.placement(s)
            c = _box_corners(lo2, hi2) @ R.T + tr
            return np.minimum(lo1, c.min(0)), np.maximum(hi1, c.max(0))
        if t == "box":
            h = np.array([g("x"), g("y"), g("z")]) * L / 2
        elif t in ("tube", "cutTube"):
            h = np.array([g("rmax"), g("rmax"), g("z") / 2]) * L
        elif t == "cone":
            r = max(g("rmax1"), g("rmax2"))
            h = np.array([r, r, g("z") / 2]) * L
        elif t == "orb":
            h = np.array([g("r")] * 3) * L
        elif t == "sphere":                         # honour the theta range (hemispheres, caps)
            A = _UNITS.get(s.get("aunit", "rad"), 1.0)
            rmin, rmax = g("rmin"), g("rmax")
            t0 = g("starttheta") * A
            t1 = t0 + g("deltatheta", math.pi / A) * A
            th = [t0, t1] + [v for v in (0.0, math.pi / 2, math.pi) if t0 < v < t1]
            zs = [r * math.cos(a) for r in (rmin, rmax) for a in th]
            rho = max(rmax * math.sin(a) for a in th)
            return np.array([-rho, -rho, min(zs)]) * L, np.array([rho, rho, max(zs)]) * L
        elif t == "ellipsoid":
            h = np.array([g("ax"), g("by"), g("cz")]) * L
        elif t == "eltube":
            h = np.array([g("dx"), g("dy"), g("dz")]) * L
        elif t == "trd":
            h = np.array([max(g("x1"), g("x2")) / 2, max(g("y1"), g("y2")) / 2, g("z") / 2]) * L
        elif t in ("trap", "arb8"):
            vals = [g(k) for k in ("x1", "x2", "x3", "x4", "y1", "y2")]
            w = max(vals[:4] + [0]) / 2 + abs(g("z")) * 0.5
            h = np.array([w, max(vals[4:] + [0]) / 2 + abs(g("z")) * 0.5, g("z") / 2]) * L
        elif t == "para":
            h = np.array([g("x") + g("y"), g("y") + g("z"), g("z")]) * L / 2
        elif t == "torus":
            r = g("rtor") + g("rmax")
            h = np.array([r, r, g("rmax")]) * L
        elif t in ("polycone", "polyhedra", "genericPolycone", "genericPolyhedra"):
            zs = [self.ev(z.get("z")) for z in s if z.tag in ("zplane", "rzpoint")] or [0.0]
            rs = [self.ev(z.get("rmax", z.get("r", 0))) for z in s if z.tag in ("zplane", "rzpoint")] or [0.0]
            r = max(rs)
            return np.array([-r, -r, min(zs)]) * L, np.array([r, r, max(zs)]) * L
        elif t == "tessellated":
            return -np.ones(3) * 1e3, np.ones(3) * 1e3
        else:
            self.unknown_solids.add(f"{s.get('name')} ({t})")
            h = np.ones(3) * 5
        return -h, h

    def lv_bbox(self, lv):
        return self.bbox(self.volumes[lv].find("solidref").get("ref"))

    # -- traversal ----------------------------------------------------------
    def walk(self, wanted):
        # Yield (lv, path, copynumber, t, R) for every placement of an LV for which wanted(lv) is True.
        stack = [(self.world, np.zeros(3), np.eye(3), (self.world,))]
        while stack:
            lv, t, R, path = stack.pop()
            for pv in self.volumes[lv].findall("physvol"):
                ref = pv.find("volumeref")
                if ref is None:
                    continue
                child = ref.get("ref")
                tl, Rl = self.placement(pv)
                tg, Rg = t + R @ tl, R @ Rl
                pvname = pv.get("name") or child + "_PV"
                if wanted(child):
                    yield child, path + (pvname,), int(self.ev(pv.get("copynumber"), 0)), tg, Rg
                if child in self.volumes:
                    stack.append((child, tg, Rg, path + (pvname,)))


def _selector(g, select, sensor_aux=("SensDet", "PD"), sensor_value="PhotonDetector"):
    if select in (None, "aux"):
        names = {n for n, v in g.volumes.items()
                 for a in v.findall("auxiliary")
                 if a.get("auxtype") in sensor_aux and a.get("auxvalue") == sensor_value}
        if not names:
            raise RuntimeError(f"{g.path}: no LV carries SensDet/PD = {sensor_value}. Pass a regex instead, "
                               f"e.g. SENSORS = r'^volOpDetSensitive'")
        return names.__contains__, f"aux SensDet/PD={sensor_value}"
    if isinstance(select, (list, tuple, set)):
        names = set(select)
        return names.__contains__, f"LV list ({len(names)})"
    rx = re.compile(select)
    return (lambda n: bool(rx.search(n))), f"regex {select!r}"


def _placements_frame(g, wanted, extra_cols=True):
    rows = []
    for lv, path, copyno, t, R in g.walk(wanted):
        lo, hi = g.lv_bbox(lv)
        c = _box_corners(lo, hi) @ R.T + t
        glo, ghi = c.min(0), c.max(0)
        rows.append((lv, "/".join(path), copyno, *((glo + ghi) / 2), *glo, *ghi, *(R @ ((lo + hi) / 2) + t)))
    cols = ["lv", "path", "copynumber", "x", "y", "z", "xmin", "ymin", "zmin", "xmax", "ymax", "zmax",
            "ox", "oy", "oz"]
    return pd.DataFrame(rows, columns=cols)


def read_gdml_sensors(gdml_path, select="aux", verbose=True):
    # One row per sensor placement: sensor_index, lv, path, copynumber, centre x/y/z, global bbox [mm].
    g = gdml_path if isinstance(gdml_path, GDML) else GDML(gdml_path)
    wanted, how = _selector(g, select)
    df = _placements_frame(g, wanted)
    if df.empty:
        raise RuntimeError(f"no placements of sensor volumes ({how}) reachable from the world volume")
    df = df.sort_values(["lv", "copynumber", "z", "y", "x"], kind="stable").reset_index(drop=True)
    df.insert(0, "sensor_index", np.arange(len(df)))
    df.attrs.update(gdml=g.path, select=how)
    if verbose:
        n_lv = df.lv.nunique()
        print(f"{Path(g.path).name}: {len(df)} sensor placements of {n_lv} LV(s) [{how}]")
        print("  LVs:", ", ".join(sorted(df.lv.unique())[:6]) + (" ..." if n_lv > 6 else ""))
        ext = df[["xmax", "ymax", "zmax"]].to_numpy() - df[["xmin", "ymin", "zmin"]].to_numpy()
        print("  sensor size [mm] (median dx, dy, dz):", np.round(np.median(ext, 0), 1))
        print("  sensor centres span [m]: x %.2f..%.2f  y %.2f..%.2f  z %.2f..%.2f" %
              tuple(v / 1e3 for a in "xyz" for v in (df[a].min(), df[a].max())))
        dup = df.copynumber.duplicated().sum()
        print(f"  copy numbers: {df.copynumber.nunique()} unique"
              + (f" -> {dup} duplicates: SensorID cannot be a copy number, use MATCH='position'" if dup else ""))
        if g.unknown_solids:
            print("  note: bounding box guessed for", sorted(g.unknown_solids)[:5])
    return df


def read_gdml_volumes(gdml_path, regex, verbose=True):
    # Global bounding boxes of all placements of LVs matching `regex` (for drawing context outlines).
    if not regex:
        return pd.DataFrame()
    g = gdml_path if isinstance(gdml_path, GDML) else GDML(gdml_path)
    rx = re.compile(regex)
    df = _placements_frame(g, lambda n: bool(rx.search(n)))
    if verbose:
        print(f"context {regex!r}: {len(df)} placements of {df.lv.nunique() if len(df) else 0} LV(s)")
    return df

## Planes and plotting helpers
`find_planes` groups sensors by their thinnest bounding-box axis and the coordinate along it (sensors
closer than `gap` mm along that axis join the same plane). Each plane is then drawn in its own two axes.

In [ ]:
_AX = "xyz"
_SCALE = {"m": 1e-3, "cm": 0.1, "mm": 1.0}


def find_planes(sensors, gap=150.0, min_size=2):
    # Adds/returns column 'plane' (int, -1 = not in any plane) and a description table.
    lo = sensors[["xmin", "ymin", "zmin"]].to_numpy()
    hi = sensors[["xmax", "ymax", "zmax"]].to_numpy()
    normal = np.argmin(hi - lo, axis=1)
    plane = np.full(len(sensors), -1)
    info, pid = [], 0
    for ax in range(3):
        idx = np.nonzero(normal == ax)[0]
        if not len(idx):
            continue
        c = sensors[_AX[ax]].to_numpy()[idx]
        order = np.argsort(c)
        cuts = np.nonzero(np.diff(c[order]) > gap)[0] + 1
        for grp in np.split(order, cuts):
            members = idx[grp]
            if len(members) < min_size:
                continue
            plane[members] = pid
            info.append(dict(plane=pid, normal=_AX[ax], coord_mm=float(np.median(c[grp])), n=len(members)))
            pid += 1
    sensors["plane"] = plane
    table = pd.DataFrame(info)
    if len(table):
        table["axes"] = table.normal.map({"x": "z,y", "y": "z,x", "z": "x,y"})
    return table


def _plane_axes(normal):
    # (horizontal, vertical) axes used to draw a plane with the given normal.
    return {"x": ("z", "y"), "y": ("z", "x"), "z": ("x", "y")}[normal]


def _rects(df, ha, va, s):
    return [Rectangle((r[f"{ha}min"] * s, r[f"{va}min"] * s), (r[f"{ha}max"] - r[f"{ha}min"]) * s,
                      (r[f"{va}max"] - r[f"{va}min"]) * s) for _, r in df.iterrows()]


def _sensor_polys(df, s=1.0, style="face"):
    # 3D polygons of every sensor (its global bounding box), in plot units.
    #   style="face": one rectangle per sensor - the box face across its thinnest axis
    #                 (the window of a flat photon detector, e.g. a DUNE arapuca)
    #   style="box" : all 6 faces of the bounding box
    # Returns (polys [M,4,3], owner [M]) where owner is the row number in df of each polygon.
    lo = df[["xmin", "ymin", "zmin"]].to_numpy(float) * s
    hi = df[["xmax", "ymax", "zmax"]].to_numpy(float) * s
    n = len(df)
    if style == "box":
        polys, owner = [], []
        for ax in range(3):
            a, b = [k for k in range(3) if k != ax]
            for side in (lo, hi):
                q = np.empty((n, 4, 3))
                q[:, :, ax] = side[:, ax, None]
                q[:, :, a] = np.stack([lo[:, a], hi[:, a], hi[:, a], lo[:, a]], 1)
                q[:, :, b] = np.stack([lo[:, b], lo[:, b], hi[:, b], hi[:, b]], 1)
                polys.append(q)
                owner.append(np.arange(n))
        return np.concatenate(polys), np.concatenate(owner)
    normal = np.argmin(hi - lo, axis=1)
    q = np.empty((n, 4, 3))
    for ax in range(3):
        m = normal == ax
        if not m.any():
            continue
        a, b = [k for k in range(3) if k != ax]
        q[m, :, ax] = ((lo[m, ax] + hi[m, ax]) / 2)[:, None]
        q[m, :, a] = np.stack([lo[m, a], hi[m, a], hi[m, a], lo[m, a]], 1)
        q[m, :, b] = np.stack([lo[m, b], lo[m, b], hi[m, b], hi[m, b]], 1)
    return q, np.arange(n)


def _view_3d(ax, vertical_axis="y", elev=20, azim=-60):
    # Camera for 3D panels. Must be called BEFORE _set_3d_limits: matplotlib converts the box aspect
    # using the vertical axis that is active when the aspect is set.
    try:
        ax.view_init(elev=elev, azim=azim, vertical_axis=vertical_axis)
    except TypeError:                         # matplotlib < 3.6: z is always up
        ax.view_init(elev=elev, azim=azim)


def _set_3d_limits(ax, sensors, s, pad=0.03):
    # Poly3DCollection does not autoscale: set limits + true aspect from the sensor boxes.
    lo = sensors[["xmin", "ymin", "zmin"]].min().to_numpy(float) * s
    hi = sensors[["xmax", "ymax", "zmax"]].max().to_numpy(float) * s
    ext = np.maximum(hi - lo, (hi - lo).max() * 0.05)
    mid = (lo + hi) / 2
    for setter, m, e in zip((ax.set_xlim, ax.set_ylim, ax.set_zlim), mid, ext):
        setter(m - e * (0.5 + pad), m + e * (0.5 + pad))
    ax.set_box_aspect(ext / ext.max())


def _draw_context(ax, context, ha, va, s, sel=None):
    if context is None or not len(context):
        return
    ctx = context if sel is None else context[sel]
    ax.add_collection(PatchCollection(_rects(ctx, ha, va, s), facecolor="none", edgecolor="0.55", lw=0.6))


def _save(fig, name):
    if SAVE_DIR:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(SAVE_DIR) / f"{name}.png", dpi=150, bbox_inches="tight")
    if not INLINE_FORMAT:
        plt.close(fig)


def _plane_aspect(sensors, planes):
    # Median width/height of the planes as drawn (used to lay out the panels).
    r = []
    for _, p in planes.iterrows():
        ha, va = _plane_axes(p.normal)
        g = sensors[sensors.plane == p.plane]
        w = g[f"{ha}max"].max() - g[f"{ha}min"].min()
        h = g[f"{va}max"].max() - g[f"{va}min"].min()
        r.append(w / max(h, 1.0))
    return float(np.median(r)) if r else 1.0


def _grid(n, ncols=2, w=7.5, h=4.2, aspect=None):
    if aspect is not None:                      # long planes (e.g. DUNE 58 m x 12 m): one per row
        ncols = 1 if aspect > 2.5 else ncols
        w = 15.0 if ncols == 1 else 7.5
        h = min(max(w / aspect * 0.9 + 1.2, 2.5), 9)
    ncols = min(ncols, n)
    nrows = math.ceil(n / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows), squeeze=False)
    for a in axs.flat[n:]:
        a.set_visible(False)
    return fig, axs.flat

## Draw every sensor (GDML)

In [ ]:
def draw_sensor_positions(sensors, context=None, unit=LENGTH_UNIT, color_by="lv", max_legend=8,
                          style="face", vertical_axis=VERTICAL_AXIS, elev=20, azim=-60):
    # 3D view of every sensor (real rectangular shape) + XY / XZ / ZY projections with sensor
    # footprints and context outlines. style="face" (sensor window) or "box" (full bounding box).
    s = _SCALE[unit]
    fig = plt.figure(figsize=(16, 11))
    ax3 = fig.add_subplot(2, 2, 1, projection="3d")
    groups = sensors.groupby(color_by) if sensors[color_by].nunique() <= max_legend else [("sensors", sensors)]
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    handles = []
    for (name, g), c in zip(groups, colors):
        polys, _ = _sensor_polys(g, s, style)
        ax3.add_collection3d(Poly3DCollection(polys, facecolor=c, edgecolor="none", alpha=0.8))
        handles.append(Rectangle((0, 0), 1, 1, color=c, label=f"{name} ({len(g)})"))
    _view_3d(ax3, vertical_axis, elev, azim)
    _set_3d_limits(ax3, sensors, s)
    ax3.set(xlabel=f"x [{unit}]", ylabel=f"y [{unit}]", zlabel=f"z [{unit}]",
            title=f"{len(sensors)} sensors - {Path(sensors.attrs.get('gdml', '')).name}")
    ax3.legend(handles=handles, fontsize=7, loc="upper left")
    for k, (ha, va) in enumerate([("x", "y"), ("z", "x"), ("z", "y")]):
        ax = fig.add_subplot(2, 2, k + 2)
        _draw_context(ax, context, ha, va, s)
        for (name, g), c in zip(groups, colors):
            ax.add_collection(PatchCollection(_rects(g, ha, va, s), facecolor=c, edgecolor="none", alpha=0.6))
        ax.autoscale_view()
        ax.set_aspect("equal", adjustable="datalim")
        ax.set(xlabel=f"{ha} [{unit}]", ylabel=f"{va} [{unit}]", title=f"{ha}-{va} projection")
    fig.tight_layout()
    _save(fig, "sensor_positions")
    return fig


def draw_sensor_planes(sensors, planes, context=None, unit=LENGTH_UNIT, ncols=2, label_every=0, max_planes=12):
    # One panel per sensor plane, sensors at their real size (coloured by LV).
    # Only the max_planes planes with the most sensors are drawn (keeps the figure small).
    s = _SCALE[unit]
    if len(planes) > max_planes:
        print(f"drawing the {max_planes} largest of {len(planes)} planes (max_planes=...)")
        planes = planes.nlargest(max_planes, "n").sort_values("plane")
    fig, axs = _grid(len(planes), ncols, aspect=_plane_aspect(sensors, planes))
    for ax, (_, p) in zip(axs, planes.iterrows()):
        ha, va = _plane_axes(p.normal)
        g = sensors[sensors.plane == p.plane]
        if context is not None and len(context):
            n = p.normal
            near = (context[f"{n}min"] <= p.coord_mm + 50) & (context[f"{n}max"] >= p.coord_mm - 50)
            _draw_context(ax, context, ha, va, s, near)
        ax.add_collection(PatchCollection(_rects(g, ha, va, s), facecolor="C0", edgecolor="C0", lw=0.3, alpha=0.7))
        if label_every:
            for _, r in g.iloc[::label_every].iterrows():
                ax.annotate(str(r.sensor_index), (r[ha] * s, r[va] * s), fontsize=5, ha="center")
        ax.autoscale_view()
        ax.set_aspect("equal", adjustable="datalim")
        ax.set(xlabel=f"{ha} [{unit}]", ylabel=f"{va} [{unit}]",
               title=f"plane {p.plane}: {p.normal} = {p.coord_mm * s:.3g} {unit}  ({p.n} sensors)")
    fig.tight_layout()
    _save(fig, "sensor_planes")
    return fig

## Hits: read and match to sensors

In [ ]:
HIT_BRANCHES = ["evtID", "hit_Id", "SensorID", "x", "y", "z", "t", "wavelength", "boundary"]


def load_hits(root_file, tree=TREE_NAME, events=None, branches=None):
    # OpticksHits / Geant4Hits -> pandas DataFrame. events: None (all), int or list of evtIDs.
    with uproot.open(root_file) as f:
        if tree not in f:
            raise KeyError(f"{tree!r} not in {root_file}; keys: {f.keys()}")
        t = f[tree]
        want = [b for b in (branches or HIT_BRANCHES) if b in t.keys()]
        df = t.arrays(want, library="pd")
    if events is not None:
        ev = [events] if np.isscalar(events) else list(events)
        df = df[df.evtID.isin(ev)]
    df = df.reset_index(drop=True)
    print(f"{Path(root_file).name}:{tree}: {len(df):,} hits in {df.evtID.nunique()} event(s)")
    return df


def match_hits(hits, sensors, how=MATCH, k=16, tol=5.0):
    # Adds 'sensor_index' (-1 = unmatched) and 'match_dist' [mm] to hits.
    #   how='position': the sensor whose global bbox contains the hit (within tol mm), otherwise the
    #                   sensor whose bbox is closest (searched among the k nearest centres)
    #   how='id'      : SensorID == copy number (only valid when copy numbers are unique)
    if how == "id":
        if sensors.copynumber.duplicated().any():
            raise ValueError("copy numbers are not unique in this GDML; use how='position'")
        lut = pd.Series(sensors.sensor_index.to_numpy(), index=sensors.copynumber)
        hits["sensor_index"] = hits.SensorID.map(lut).fillna(-1).astype(int)
    else:
        P = hits[["x", "y", "z"]].to_numpy(float)
        lo = sensors[["xmin", "ymin", "zmin"]].to_numpy()
        hi = sensors[["xmax", "ymax", "zmax"]].to_numpy()
        tree = cKDTree(sensors[["x", "y", "z"]].to_numpy())
        k = min(k, len(sensors))
        _, cand = tree.query(P, k=k)
        cand = cand.reshape(len(P), k)
        d = np.linalg.norm(np.maximum(0, np.maximum(lo[cand] - P[:, None], P[:, None] - hi[cand])), axis=2)
        best = np.argmin(d, axis=1)
        hits["sensor_index"] = cand[np.arange(len(P)), best]
        hits["match_dist"] = d[np.arange(len(P)), best]
        far = hits.match_dist > tol
        n_far = int(far.sum())
        print(f"matched {len(hits) - n_far:,} hits inside a sensor box (<= {tol} mm); "
              f"{n_far:,} further away (median {hits.match_dist[far].median() if n_far else 0:.1f} mm) "
              "kept on the nearest sensor")
    if "SensorID" in hits and how != "id" and not sensors.copynumber.duplicated().any():
        agree = (hits.SensorID.to_numpy() == sensors.copynumber.to_numpy()[hits.sensor_index]).mean()
        print(f"SensorID == copy number of the matched sensor for {agree:.1%} of hits")
    return hits


def hits_per_sensor(hits, sensors, per_event=False):
    n = np.bincount(hits.sensor_index[hits.sensor_index >= 0], minlength=len(sensors)).astype(float)
    return n / max(hits.evtID.nunique(), 1) if per_event else n

## Heat maps

In [ ]:
def _norm(counts, log, vmax=None):
    hit = counts > 0
    vmax = vmax or (counts.max() if hit.any() else 1)
    if log:
        return LogNorm(vmin=max(counts[hit].min(), 1e-3) if hit.any() else 0.1, vmax=max(vmax, 1e-3))
    return Normalize(0, vmax)


def plot_hit_heatmap_planes(sensors, planes, counts, context=None, unit=LENGTH_UNIT, log=LOG_COLOR,
                            cmap="viridis", zero_color="0.85", only_hit_planes=True, max_planes=12,
                            ncols=2, title="hits per sensor", save_as="hit_heatmap_planes"):
    # One panel per plane; every sensor drawn at its real size, coloured by its hit count.
    counts = np.asarray(counts, float)
    s = _SCALE[unit]
    per_plane = planes.assign(hits=[counts[sensors.plane.to_numpy() == p].sum() for p in planes.plane])
    if only_hit_planes:
        per_plane = per_plane[per_plane.hits > 0]
    per_plane = per_plane.sort_values("hits", ascending=False).head(max_planes).sort_values("plane")
    if per_plane.empty:
        print("no hits on any plane")
        return None
    norm = _norm(counts, log)
    fig, axs = _grid(len(per_plane), ncols, aspect=_plane_aspect(sensors, per_plane))
    for ax, (_, p) in zip(axs, per_plane.iterrows()):
        ha, va = _plane_axes(p.normal)
        m = sensors.plane.to_numpy() == p.plane
        g, c = sensors[m], counts[m]
        if context is not None and len(context):
            n = p.normal
            near = (context[f"{n}min"] <= p.coord_mm + 50) & (context[f"{n}max"] >= p.coord_mm - 50)
            _draw_context(ax, context, ha, va, s, near)
        ax.add_collection(PatchCollection(_rects(g[c == 0], ha, va, s), facecolor=zero_color, edgecolor="none"))
        lit = PatchCollection(_rects(g[c > 0], ha, va, s), cmap=cmap, norm=norm, edgecolor="none")
        lit.set_array(c[c > 0])
        ax.add_collection(lit)
        ax.autoscale_view()
        ax.set_aspect("equal", adjustable="datalim")
        ax.set(xlabel=f"{ha} [{unit}]", ylabel=f"{va} [{unit}]",
               title=f"plane {p.plane} ({p.normal} = {p.coord_mm * s:.3g} {unit}): {int(p.hits):,} hits")
        fig.colorbar(lit, ax=ax, shrink=0.8, label="hits")
    fig.suptitle(f"{title}: {counts.sum():,.0f} hits on {(counts > 0).sum()}/{len(counts)} sensors")
    fig.tight_layout()
    _save(fig, save_as)
    return fig


def plot_hit_heatmap_3d(sensors, counts, unit=LENGTH_UNIT, log=LOG_COLOR, cmap="viridis", zero_color="0.85",
                        style="face", show_empty=True, empty_alpha=0.25,
                        vertical_axis=VERTICAL_AXIS, elev=20, azim=-60,
                        title="hits per sensor (3D)", save_as="hit_heatmap_3d"):
    # 3D heat map: every sensor drawn with its real rectangular shape (global bounding box),
    # coloured by its hit count. style="face" draws the sensor window, style="box" the full box.
    counts = np.asarray(counts, float)
    s, hit = _SCALE[unit], counts > 0
    norm, cm = _norm(counts, log), plt.get_cmap(cmap)
    fig = plt.figure(figsize=(11, 8))
    ax = fig.add_subplot(projection="3d")
    if show_empty and (~hit).any():
        polys, _ = _sensor_polys(sensors[~hit], s, style)
        ax.add_collection3d(Poly3DCollection(polys, facecolor=zero_color, edgecolor="none", alpha=empty_alpha))
    if hit.any():
        polys, owner = _sensor_polys(sensors[hit], s, style)
        ax.add_collection3d(Poly3DCollection(polys, facecolors=cm(norm(counts[hit][owner])), edgecolor="none"))
    _view_3d(ax, vertical_axis, elev, azim)
    _set_3d_limits(ax, sensors, s)
    ax.set(xlabel=f"x [{unit}]", ylabel=f"y [{unit}]", zlabel=f"z [{unit}]",
           title=f"{title}: {counts.sum():,.0f} hits on {hit.sum()}/{len(counts)} sensors")
    fig.colorbar(ScalarMappable(norm=norm, cmap=cm), ax=ax, shrink=0.6, pad=0.12, label="hits")
    _save(fig, save_as)
    return fig


def plot_hit_projections(hits, sensors=None, unit=LENGTH_UNIT, bins=200, log=True, save_as="hit_projections"):
    # 2D histograms of the raw hit positions in XY, ZX, ZY (independent of any sensor matching).
    s = _SCALE[unit]
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (ha, va) in zip(axs, [("x", "y"), ("z", "x"), ("z", "y")]):
        rng = None
        if sensors is not None:
            rng = [[sensors[f"{ha}min"].min() * s, sensors[f"{ha}max"].max() * s],
                   [sensors[f"{va}min"].min() * s, sensors[f"{va}max"].max() * s]]
        h = ax.hist2d(hits[ha] * s, hits[va] * s, bins=bins, range=rng, cmin=1,
                      norm=LogNorm() if log else None, cmap="viridis")
        fig.colorbar(h[3], ax=ax, label="hits")
        ax.set(xlabel=f"{ha} [{unit}]", ylabel=f"{va} [{unit}]", title=f"hit positions {ha}-{va}")
    fig.tight_layout()
    _save(fig, save_as)
    return fig


def check_matching(hits, bins=100, save_as="hit_checks"):
    # Distance of each hit to its sensor box + hit time / wavelength spectra.
    fig, axs = plt.subplots(1, 3, figsize=(16, 3.8))
    if "match_dist" in hits:
        axs[0].hist(np.clip(hits.match_dist, 0, None), bins=bins)
        axs[0].set_yscale("log")
        axs[0].set(xlabel="distance hit -> sensor box [mm]", ylabel="hits", title="matching")
    for ax, col, lab in ((axs[1], "t", "t [ns]"), (axs[2], "wavelength", "wavelength [nm]")):
        if col in hits:
            ax.hist(hits[col], bins=bins)
            ax.set(xlabel=lab, ylabel="hits")
    fig.tight_layout()
    _save(fig, save_as)
    return fig

## 1. Sensors from the GDML file

In [ ]:
gdml = GDML(GDML_FILE)
sensors = read_gdml_sensors(gdml, select=SENSORS)
context = read_gdml_volumes(gdml, CONTEXT)
planes = find_planes(sensors)
print(f"{len(planes)} planes; {(sensors.plane < 0).sum()} sensors in no plane")
planes

In [ ]:
draw_sensor_positions(sensors, context);

In [ ]:
draw_sensor_planes(sensors, planes, context);

## 2. Hits

In [ ]:
hits = load_hits(ROOT_FILE, events=EVENTS)
hits = match_hits(hits, sensors, how=MATCH)
counts = hits_per_sensor(hits, sensors)
hits.head()

## 3. Heat maps — all selected events

In [ ]:
plot_hit_heatmap_planes(sensors, planes, counts, context);

In [ ]:
plot_hit_heatmap_3d(sensors, counts);

In [ ]:
plot_hit_projections(hits, sensors);

## 4. One event at a time

In [ ]:
for evt in sorted(hits.evtID.unique())[:EVENT_DISPLAYS]:
    c = hits_per_sensor(hits[hits.evtID == evt], sensors)
    plot_hit_heatmap_planes(sensors, planes, c, context, title=f"event {evt}", max_planes=4,
                            save_as=f"hit_heatmap_planes_evt{evt}")

In [ ]:
check_matching(hits);